<img src="logo.png" alt="Vegeta" width="240">

# Svarog ablation — the target and the shields of notebook 46 as FEniCSx transients (book `svarog`)

Notebook `46_svarog_remote` says, in closed form, *when* the target melts under N beams crossing at 20, 50 and 100 m and
*whether* a shield holds against a stray beam. This notebook sets the field solver against those forms: book **`svarog`**
(`wf list`: **S1 `target-ablation`**, **S2 `shield-ablation`**; variants = the standoff `20m`, `50m`, `100m`;
`workflows/svarog/ablation_fenicsx.py`). Each case is an axisymmetric r–z transient in FEniCSx — conduction with the
apparent-heat-capacity enthalpy method (melting, then the removal plateau: boiling for the metals, decomposition for
the plastic), convection and radiation at the faces, implicit Euler with Picard iterations, ablation by cell
deactivation. The beam each case sees is the budget's at that standoff from 46's head: the laser's properties as the
distance modifies them.

**Nothing is solved here.** The runs are the engineer's to launch (`wf S1 --variant 20m --detach`, …; dolfinx from
conda-forge `fenics-dolfinx` or the `dolfinx/dolfinx` Docker image); this notebook prints the commands of the missing
ones, launches them only when asked (`SVAROG_LAUNCH=1`), and reads whatever runs exist — NOT RUN otherwise, the closed
forms standing alone.

**Its contract** (`flow.notebook("47_svarog_ablation", …)` in `workflows/flow.py`): takes notebook 46's report and the
six runs, all optional; gives `designs/data/reports/47_svarog_ablation.json`.

In [1]:
DEBUG = False
from vegeta import mock
DEBUG = mock.setup(DEBUG)

## 1. Prerequisites

The gate: notebook 46's report (the head and its efficiency) and the six runs, each with its status and the command
that makes it.

In [2]:
import json, math, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import yanus
sys.path.insert(0, "designs")
import svarog_remote as sr

pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
OUT = Path("output/47_svarog_ablation"); OUT.mkdir(parents=True, exist_ok=True)
GATE = yanus.enter("47_svarog_ablation", debug=DEBUG)
display(GATE.table())
REPORT_KEY = "46_svarog_remote:notebooks/designs/data/svarog_remote.json"
R46 = json.loads(Path(GATE.inputs[REPORT_KEY]).read_text()) if GATE.inputs.get(REPORT_KEY) else None
HEAD_POWER = 1000.0
W_BEAM = (R46["head"]["beam_radius_at_L2_mm"] * 1e-3) if R46 else sr.HEAD_W_IN
EFF = {20.0: R46["head"]["peak_W_cm2_per_kW"] / sr.spot_at(sr.LASERS["IPG YLR-1000-SM"], W_BEAM, 20.0)["I0_W_cm2"]} if R46 else {20.0: sr.HEAD_EFFICIENCY}
EFF.update({50.0: EFF[20.0], 100.0: EFF[20.0]})
print("notebook 46's head:", f"w = {W_BEAM*1e3:.1f} mm at L2, efficiency {EFF[20.0]:.2f}" + ("" if R46 else " (46's report missing: the module's defaults)"))
rng = np.random.default_rng(11)
SCEN = {(n, P, Lx): sr.scenario(n, sr.SOURCES_BY_POWER[P], Lx, w_in=W_BEAM, efficiency=EFF[Lx], rng=rng)
        for Lx in (20.0, 50.0, 100.0) for n, P in ((2, 1000.0), (10, 1000.0), (10, 2000.0), (20, 1000.0), (10, 10000.0))}
REPORT = {"head_w_beam_m": W_BEAM, "efficiency": EFF[20.0]}

,role,selection,status,provenance,compatibility,usable,fix
input,,,,,,,
svarog_remote.json (46_svarog_remote),optional,notebooks/designs/data/svarog_remote.json,on file (2026-10-18T10:02:24Z),46_svarog_remote 2026-10-18T10:02 (git c5fdbd85b9d3),,True,scripts/run_notebooks.sh 46_svarog_remote.ipynb
"S1 target-ablation, 20m",optional,—,not pinned,,,False,wf S1 --variant 20m
"S1 target-ablation, 50m",optional,—,not pinned,,,False,wf S1 --variant 50m
"S1 target-ablation, 100m",optional,—,not pinned,,,False,wf S1 --variant 100m
"S2 shield-ablation, 20m",optional,—,not pinned,,,False,wf S2 --variant 20m
"S2 shield-ablation, 50m",optional,—,not pinned,,,False,wf S2 --variant 50m
"S2 shield-ablation, 100m",optional,—,not pinned,,,False,wf S2 --variant 100m


notebook 46's head: w = 20.8 mm at L2, efficiency 0.59


## 2. The six runs — the commands, and the launch when asked

The closed forms above say *when* the target melts and *whether* a shield holds; the field solver says how the front
moves. Book **`svarog`** (`workflows/svarog/ablation_fenicsx.py`, `wf list`) solves each case as an axisymmetric r–z
transient in FEniCSx: conduction with the apparent-heat-capacity enthalpy method (melting, then the removal plateau —
boiling for the metals, decomposition for the plastic), convection and radiation at the faces, implicit Euler with
Picard iterations, ablation by cell deactivation. Its **variants are the standoffs** — `20m`, `50m`, `100m` — and
the beam each case sees is the budget's at that standoff from notebook 46's head (its §3.1, §4.1): the laser's
properties as the distance modifies them. **S1 `target-ablation`**: the steel target under each set's summed beams
(2 × 1, 10 × 1, 10 × 2, 20 × 1 kW single-mode, 10 × 10 kW multimode) for 2 s — melt onset, boiling onset (the
keyhole), the ablated depth. **S2 `shield-ablation`**: aluminium, steel and black polycarbonate at 1, 3 and 5 mm under
one stray kilowatt 20, 50 and 100 m past the focus — surface melting and burn-through times, against notebook 46's §6.1.

The gate took the six runs (two chapters × three standoffs) as optional inputs. The cell below prints the command
of every missing run (`wf S1 --variant 20m --detach`, …) for the engineer to launch — and launches them itself,
detached, only when asked (`SVAROG_LAUNCH=1` in the environment) and dolfinx is importable (conda-forge
`fenics-dolfinx`, or the `dolfinx/dolfinx` Docker image). The runs are read when the notebook is next executed;
nothing is computed in place, and nothing is launched by default.

In [3]:
import importlib.util, shlex, subprocess
ROOT = Path("..").resolve()
LAUNCH = os.environ.get("SVAROG_LAUNCH", "0") == "1"             # opt in: SVAROG_LAUNCH=1 launches the missing runs detached; the default only prints their commands
HAVE_DOLFINX = importlib.util.find_spec("dolfinx") is not None
ABL_ROWS = [r for r in GATE.rows if r.key and r.key.startswith("svarog/")]
display(pd.DataFrame([{"run": r.input, "status": r.status, "usable": r.usable, "selection": r.selection or "", "fix": r.fix or ""} for r in ABL_ROWS]))
launched = []
for r in ABL_ROWS:
    if r.usable:
        continue
    chapter, variant = r.key.split("/", 1)[1].split("@")
    campaign = {"target-ablation": "target", "shield-ablation": "shield"}[chapter]
    cmd = [sys.executable, str(ROOT / "workflows" / "svarog" / "ablation_fenicsx.py"), "--variant", variant, "--chapter", campaign, "--quiet"]
    shown = shlex.join([".venv/bin/python", "workflows/svarog/ablation_fenicsx.py", "--variant", variant, "--chapter", campaign])
    if LAUNCH and HAVE_DOLFINX:
        out = ROOT / "workflows" / "runs"; out.mkdir(parents=True, exist_ok=True)
        log = out / f"S_{chapter}_{variant}_{time.strftime('%Y%m%d-%H%M%S')}.out"
        with log.open("ab") as f:
            proc = subprocess.Popen(cmd, cwd=ROOT, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True)
        launched.append((r.input, proc.pid, log.relative_to(ROOT)))
        print(f"launched (pid {proc.pid}): $ {shown}   → {log.relative_to(ROOT)}")
    else:
        print(f"NOT RUN ({'dolfinx not installed' if not HAVE_DOLFINX else 'not launched (SVAROG_LAUNCH=1 to launch from here)'}): $ {shown}")
if launched:
    print(f"{len(launched)} run(s) started detached; watch them with `wf runs` / `wf tail`, then run this notebook again to read them.")
REPORT["ablation_runs"] = {"launched": [(a, b, str(c)) for a, b, c in launched], "usable": [r.key for r in ABL_ROWS if r.usable], "dolfinx": HAVE_DOLFINX}

,run,status,usable,selection,fix
0,"S1 target-ablation, 20m",not pinned,False,—,wf S1 --variant 20m
1,"S1 target-ablation, 50m",not pinned,False,—,wf S1 --variant 50m
2,"S1 target-ablation, 100m",not pinned,False,—,wf S1 --variant 100m
3,"S2 shield-ablation, 20m",not pinned,False,—,wf S2 --variant 20m
4,"S2 shield-ablation, 50m",not pinned,False,—,wf S2 --variant 50m
5,"S2 shield-ablation, 100m",not pinned,False,—,wf S2 --variant 100m


NOT RUN (dolfinx not installed): $ .venv/bin/python workflows/svarog/ablation_fenicsx.py --variant 20m --chapter target
NOT RUN (dolfinx not installed): $ .venv/bin/python workflows/svarog/ablation_fenicsx.py --variant 50m --chapter target
NOT RUN (dolfinx not installed): $ .venv/bin/python workflows/svarog/ablation_fenicsx.py --variant 100m --chapter target
NOT RUN (dolfinx not installed): $ .venv/bin/python workflows/svarog/ablation_fenicsx.py --variant 20m --chapter shield
NOT RUN (dolfinx not installed): $ .venv/bin/python workflows/svarog/ablation_fenicsx.py --variant 50m --chapter shield
NOT RUN (dolfinx not installed): $ .venv/bin/python workflows/svarog/ablation_fenicsx.py --variant 100m --chapter shield


## 3. Reading the runs: the target (S1) against the closed-form dwell times, the shields (S2) against §6.1

Each usable run's cases, next to the estimate notebook 46 made without them (recomputed here from `svarog_remote`, seconds). A run that is not there is NOT RUN
and the estimate stands alone, labelled as such.

In [4]:
def _cases(key):
    s = GATE.inputs.get(key)
    return (s or {}).get("cases", {}) if s else {}
rows = []
for v in ("20m", "50m", "100m"):
    Lx = float(v[:-1])
    cs = _cases(f"svarog/target-ablation@{v}")
    for name, (n, P) in sr.TARGET_SETS.items():
        rec = cs.get(name, {})
        m = rec.get("metrics", {})
        sc = SCEN[(n, P, Lx)]
        I_abs = sr.STEEL.absorptivity * sc["peak, all aligned [W/cm²]"] * 1e4
        est = sr.dwell_times(sr.STEEL, I_abs, sc["w0 [mm]"] * 1e-3, sc["pool width [mm]"] * 1e-3)
        rows.append({"L [m]": Lx, "set": name, "regime (closed form)": sc["regime"], "melt onset, closed form [ms]": f"{est['melt onset [ms]']:.3g}",
                     "FEniCSx status": rec.get("status", "NOT RUN"), "FEniCSx melt onset [ms]": f"{m['melt_onset_s']*1e3:.3g}" if m.get("melt_onset_s") == m.get("melt_onset_s") and m else "",
                     "FEniCSx boiling onset [ms]": f"{m['removal_onset_s']*1e3:.3g}" if m and m.get("removal_onset_s") == m.get("removal_onset_s") else "",
                     "ablated depth at 2 s [mm]": f"{m['ablated_depth_end_m']*1e3:.2f}" if m else "", "surface T at 2 s [K]": f"{m['T_top_end_K']:.0f}" if m else ""})
TARGET_ABL = pd.DataFrame(rows); display(TARGET_ABL)

rows = []
for v in ("20m", "50m", "100m"):
    Lx = float(v[:-1])
    cs = _cases(f"svarog/shield-ablation@{v}")
    for key, sh_abl in sr.ABLATION_SOLIDS.items():
        sh = {"mild_steel": sr.SHIELDS[1], "aluminium": sr.SHIELDS[0], "black_pc": sr.SHIELDS[2]}[key]
        for t in sr.SHIELD_THICKNESS:
            for d in sr.SHIELD_PAST_FOCUS:
                name = f"{key}_{t*1e3:g}mm_{d:g}m"
                rec = cs.get(name, {}); m = rec.get("metrics", {})
                w0x = SCEN[(10, 1000.0, Lx)]["w0 [mm]"] * 1e-3
                est = sr.shield_response(sh, HEAD_POWER * sr.HEAD_EFFICIENCY, sr.stray_radius(W_BEAM, Lx, d, w0x), thickness=t)
                rows.append({"L [m]": Lx, "shield": sh_abl.name, "t [mm]": t * 1e3, "m past focus": d, "closed form: holds": bool(est["holds"]), "closed form: s to the limit": round(est["t_to_limit_s"], 1),
                             "FEniCSx status": rec.get("status", "NOT RUN"), "FEniCSx melt onset [s]": f"{m['melt_onset_s']:.3g}" if m and m.get("melt_onset_s") == m.get("melt_onset_s") else "",
                             "FEniCSx burn-through [s]": f"{m['burn_through_s']:.3g}" if m and m.get("burn_through_s") == m.get("burn_through_s") else ("no burn-through in the run" if m else "")})
SHIELD_ABL = pd.DataFrame(rows)
display(SHIELD_ABL[SHIELD_ABL["FEniCSx status"] != "NOT RUN"] if (SHIELD_ABL["FEniCSx status"] != "NOT RUN").any() else SHIELD_ABL.head(9))
n_run = int((SHIELD_ABL["FEniCSx status"] != "NOT RUN").sum()) + int((TARGET_ABL["FEniCSx status"] != "NOT RUN").sum())
print(f"{n_run} FEniCSx case(s) read" + ("" if n_run else " — every row is NOT RUN: the closed forms stand alone until the runs exist"))
REPORT["ablation"] = {"target": TARGET_ABL.to_dict("records"), "shield_cases_read": n_run}

,L [m],set,regime (closed form),"melt onset, closed form [ms]",FEniCSx status,FEniCSx melt onset [ms],FEniCSx boiling onset [ms],ablated depth at 2 s [mm],surface T at 2 s [K]
0,20.0,2x1kW,conduction,0.0575,NOT RUN,,,,
1,20.0,10x1kW,keyhole,0.0023,NOT RUN,,,,
2,20.0,10x2kW,keyhole,0.00112,NOT RUN,,,,
3,20.0,20x1kW,keyhole,0.000575,NOT RUN,,,,
4,20.0,10x10kW,conduction,0.0446,NOT RUN,,,,
5,50.0,2x1kW,heating only,2.25,NOT RUN,,,,
6,50.0,10x1kW,conduction,0.0899,NOT RUN,,,,
7,50.0,10x2kW,conduction,0.0438,NOT RUN,,,,
8,50.0,20x1kW,conduction,0.0225,NOT RUN,,,,
9,50.0,10x10kW,conduction,1.74,NOT RUN,,,,


,L [m],shield,t [mm],m past focus,closed form: holds,closed form: s to the limit,FEniCSx status,FEniCSx melt onset [s],FEniCSx burn-through [s]
0,20.0,mild steel,1.0,20.0,False,6.1,NOT RUN,,
1,20.0,mild steel,1.0,50.0,False,37.6,NOT RUN,,
2,20.0,mild steel,1.0,100.0,True,149.4,NOT RUN,,
3,20.0,mild steel,3.0,20.0,False,18.4,NOT RUN,,
4,20.0,mild steel,3.0,50.0,True,112.8,NOT RUN,,
5,20.0,mild steel,3.0,100.0,True,448.3,NOT RUN,,
6,20.0,mild steel,5.0,20.0,True,30.7,NOT RUN,,
7,20.0,mild steel,5.0,50.0,True,188.1,NOT RUN,,
8,20.0,mild steel,5.0,100.0,True,747.1,NOT RUN,,


0 FEniCSx case(s) read — every row is NOT RUN: the closed forms stand alone until the runs exist


In [5]:
# the fronts, when the runs exist: the ablated depth at the centre against time, one figure per usable target run
for v in ("20m", "50m", "100m"):
    sel = GATE.selections.get(f"svarog/target-ablation@{v}")
    if not sel:
        continue
    fig, ax = plt.subplots(figsize=(10, 6))
    for name in sr.TARGET_SETS:
        p = Path(sel) / "cases" / name / "result.json"
        if p.exists():
            r = json.loads(p.read_text())
            ax.plot(np.array(r["log"]["t"]) * 1e3, np.array(r["log"]["ablated_depth_centre"]) * 1e3, lw=2, label=name)
    ax.set(xlabel="t [ms]", ylabel="ablated depth at the centre [mm]", title=f"S1 target-ablation at {v}: the removal front (FEniCSx)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    plt.tight_layout(); fig.savefig(OUT / f"ablation_target_{v}.png", dpi=130); plt.show()

## 4. Reading it

- Where the FEM and the closed forms agree (melt onset within a factor of two, a shield's burn-through within ~30 %
  of the adiabatic time to the limit), notebook 46's verdicts stand; where they do not, it is the grid or the step
  (`nz`, `dt` in `svarog_remote.ablation_cases`), not the physics, and the fix is there.
- What the FEM adds that the forms cannot: the removal front's speed (the keyhole's depth against time at the target),
  a sheet thinner than the beam with its real radial conduction, and — next — a beam at an angle and a moving beam.

In [6]:
REPORT["runs"] = {r.key: {"status": r.status, "usable": r.usable, "selection": str(r.selection or "")} for r in GATE.rows if r.key and r.key.startswith("svarog/")}
path = Path("designs/data/reports/47_svarog_ablation.json"); path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(REPORT, indent=1, default=float))
STAMPS = GATE.leave()
print(f"written {path}; {len(STAMPS)} output(s) stamped")

written designs/data/reports/47_svarog_ablation.json; 1 output(s) stamped
